<a href="https://colab.research.google.com/github/marghistani22/Data_Engineering/blob/main/ConvNnetHW5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CMSC 352: Image Classification with CNNs

The notebook explores image classification with fully connected networks.  You then compare this to a well-trained convolutional neural network (CNN).  You also will attempt to optimize the CNN.

Finally, in a later part not found here, you will use transfer learning to train on custom images.

The TODO sections indicate tasks for you to complete.  Please answer questions and provide analysis by writing well organized markdown cells.

In [1]:
%matplotlib inline
from matplotlib import pyplot as plt
import numpy as np
import collections

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.set_printoptions(edgeitems=2)
torch.manual_seed(123)

# OK, now we make sure to use the CPU.
torch.device('cpu')

device(type='cpu')

In [2]:
class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

In [3]:
# Mount your google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Use the folder image at left to select a path to store the cifar image data (170MB)
# If the data is there, no download will be required.
data_path = '/content/drive/MyDrive/Classroom/CMSC352S26 Spring 2026/notes/09nnet/data/cifar10'

from torchvision import datasets, transforms
data_path = data_path
cifar10 = datasets.CIFAR10(
    data_path, train=True, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4915, 0.4823, 0.4468),
                             (0.2470, 0.2435, 0.2616))
    ]))

In [5]:
# std normalizations were previously found with this:
imgs = torch.stack([img_t for img_t, _ in cifar10], dim=3)
print(imgs.shape)
imgs.view(3, -1).mean(dim=1)
imgs.view(3, -1).std(dim=1)

torch.Size([3, 32, 32, 50000])


tensor([1.0001, 0.9999, 1.0000])

In [6]:
# validation data
cifar10_val = datasets.CIFAR10(
    data_path, train=False, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4915, 0.4823, 0.4468),
                             (0.2470, 0.2435, 0.2616))
    ]))

## Limit our attention to cats and dogs; relabel the data.  We will only compare two types of image.

In [7]:
# just two classes for cat and dog
label_map = {3: 0, 5: 1}
class_names = ['cat', 'dog']
cifar2 = [(img, label_map[label])
          for img, label in cifar10
          if label in [3,5]]
cifar2_val = [(img, label_map[label])
              for img, label in cifar10_val
              if label in [3,5]]

In [8]:
# We start with a very simple fully connected model.
first_model = nn.Sequential(
                nn.Linear(3072, 512),
                nn.Tanh(),
                nn.Linear(512, 2),
                nn.LogSoftmax(dim=1))

In [9]:
numel_list = [p.numel() for p in first_model.parameters()]
sum(numel_list), numel_list

(1574402, [1572864, 512, 1024, 2])

In [10]:
# A second BIGGER network (all to all connections)
second_model = nn.Sequential(
            nn.Linear(3072, 1024),
            nn.Tanh(),
            nn.Linear(1024, 512),
            nn.Tanh(),
            nn.Linear(512, 128),
            nn.Tanh(),
            nn.Linear(128, 2),
            nn.LogSoftmax(dim=1))

In [11]:
# Examine the number of parameters that are trainable
numel_list = [p.numel()
              for p in second_model.parameters()
              if p.requires_grad == True]
sum(numel_list), numel_list

(3737474, [3145728, 1024, 524288, 512, 65536, 128, 256, 2])

Compare the number of parmeters in the two models.


The combination of nn.LogSoftmax and nn.NLLLoss is equivalent to using nn.CrossEntropyLoss.

In [12]:
# Here is basic code to train the model using a DataLoader
# This enables the use of batches.  Note: We are not yet using a GPU.
import torch
import torch.nn as nn
import torch.optim as optim

train_loader = torch.utils.data.DataLoader(cifar2, batch_size=64,
                                           shuffle=True)

model = first_model

learning_rate = 1e-2

optimizer = optim.SGD(model.parameters(), lr=learning_rate)

loss_fn = nn.NLLLoss()

n_epochs = 3

for epoch in range(n_epochs):
    for imgs, labels in train_loader:
        outputs = model(imgs.view(imgs.shape[0], -1))
        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("Epoch: %d, Loss: %f" % (epoch, float(loss)))

/tmp/ipykernel_2712/666275500.py:29: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("Epoch: %d, Loss: %f" % (epoch, float(loss)))


Epoch: 0, Loss: 0.658918
Epoch: 1, Loss: 0.647893
Epoch: 2, Loss: 0.634012


In [13]:
# how did the training set do?
correct = 0
total = 0

with torch.no_grad():
    for imgs, labels in train_loader:
        outputs = model(imgs.view(imgs.shape[0], -1))
        _, predicted = torch.max(outputs, dim=1)
        total += labels.shape[0]
        correct += int((predicted == labels).sum())

print("Accuracy: %f" % (correct / total))

Accuracy: 0.618800


In [14]:
# evaluate on validation set
val_loader = torch.utils.data.DataLoader(cifar2_val, batch_size=64,
                                         shuffle=False)

correct = 0
total = 0

with torch.no_grad():
    for imgs, labels in val_loader:
        outputs = model(imgs.view(imgs.shape[0], -1))
        _, predicted = torch.max(outputs, dim=1)
        total += labels.shape[0]
        correct += int((predicted == labels).sum())

print("Accuracy: %f" % (correct / total))


Accuracy: 0.607000


# TODO

## 1. What should we conclude about our training so far?  Is this first_model training complete or not?  How do you know?  

The training of first_model is not complete. The model was trained for only 3 epochs, and the loss continued to decrease from 0.658918 to 0.634012. This indicates that the model was still learning and had not yet converged. In addition, the training accuracy was 61.88% and the validation accuracy was 60.70%, which shows that there was still room for improvement.

## 2. If it's not complete, then complete it!

## 3. Now train the second model.  Can you get it to generalize better than the first model?

Q2. If it’s not complete, then complete it

In [15]:
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=64, shuffle=True)

model = first_model
optimizer = optim.SGD(model.parameters(), lr=1e-2)
loss_fn = nn.NLLLoss()

n_epochs = 20

for epoch in range(n_epochs):
    for imgs, labels in train_loader:
        outputs = model(imgs.view(imgs.shape[0], -1))
        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("Epoch:", epoch + 1, "Loss:", float(loss))

Epoch: 1 Loss: 0.5749074220657349
Epoch: 2 Loss: 0.6227731108665466
Epoch: 3 Loss: 0.567913830280304
Epoch: 4 Loss: 0.7422186136245728
Epoch: 5 Loss: 0.5268387198448181
Epoch: 6 Loss: 0.8249969482421875
Epoch: 7 Loss: 0.5015230178833008
Epoch: 8 Loss: 0.5731256604194641
Epoch: 9 Loss: 0.5049297213554382
Epoch: 10 Loss: 0.586446225643158
Epoch: 11 Loss: 0.45433083176612854
Epoch: 12 Loss: 0.5401490926742554
Epoch: 13 Loss: 0.611547589302063
Epoch: 14 Loss: 0.7950103282928467
Epoch: 15 Loss: 0.46431398391723633
Epoch: 16 Loss: 0.49544671177864075
Epoch: 17 Loss: 0.401611328125
Epoch: 18 Loss: 0.40242093801498413
Epoch: 19 Loss: 0.6107779145240784
Epoch: 20 Loss: 0.38588741421699524


Q3. Now train the second model. Can you get it to generalize better than the first model?

The second model has more layers and many more trainable parameters than the first model, so it has greater capacity to learn more complex patterns. However, a larger model does not always generalize better, because it may also overfit the training data. The best way to determine whether it generalizes better is to compare its validation accuracy with that of the first model. Since the first model achieved about 60.7% validation accuracy, the second model would generalize better only if it achieves a higher validation score

In [16]:
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=64, shuffle=True)

model = second_model
optimizer = optim.SGD(model.parameters(), lr=1e-2)
loss_fn = nn.NLLLoss()

n_epochs = 20

for epoch in range(n_epochs):
    for imgs, labels in train_loader:
        outputs = model(imgs.view(imgs.shape[0], -1))
        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("Epoch:", epoch + 1, "Loss:", float(loss))

Epoch: 1 Loss: 0.7556262612342834
Epoch: 2 Loss: 0.5872995853424072
Epoch: 3 Loss: 0.6626477241516113
Epoch: 4 Loss: 0.6539474129676819
Epoch: 5 Loss: 0.5885032415390015
Epoch: 6 Loss: 0.7638992071151733
Epoch: 7 Loss: 0.6315085291862488
Epoch: 8 Loss: 0.6534464359283447
Epoch: 9 Loss: 0.7055604457855225
Epoch: 10 Loss: 0.6449930667877197
Epoch: 11 Loss: 0.7458199858665466
Epoch: 12 Loss: 0.6650665402412415
Epoch: 13 Loss: 0.706087052822113
Epoch: 14 Loss: 0.778622567653656
Epoch: 15 Loss: 0.6440659165382385
Epoch: 16 Loss: 0.5880032181739807
Epoch: 17 Loss: 0.6393092274665833
Epoch: 18 Loss: 0.555593729019165
Epoch: 19 Loss: 0.5735385417938232
Epoch: 20 Loss: 0.5559523701667786


# Convolutional Model Comparison

Here you develop a convolutional model and compare its performance to the fully connected model.


In [27]:
cnn_model = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.Tanh(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 8, kernel_size=3, padding=1),
            nn.Tanh(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(8 * 8 * 8, 32),
            nn.Tanh(),
            nn.Linear(32, 2),
            nn.LogSoftmax(dim=1))

In [ ]:
numel_list = [p.numel() for p in cnn_model.parameters()]
sum(numel_list), numel_list

(18090, [432, 16, 1152, 8, 16384, 32, 64, 2])

# TODO

## 4. Compare the number of parameters of the two fully connected models and this CNN.

In [17]:
# OK, now we make sure to use the CPU.
torch.device('cpu')

device(type='cpu')

In [28]:
# Training without using the GPU
import datetime  # <1>

def training_loop(n_epochs, optimizer, model, loss_fn, train_loader):
    for epoch in range(1, n_epochs + 1):  # over all epochs
        loss_train = 0.0
        for imgs, labels in train_loader:  # for images and labels

            outputs = model(imgs)
            loss = loss_fn(outputs, labels)

            optimizer.zero_grad()  # reset grads
            loss.backward()  # backprop
            optimizer.step()  # one step of learning
            loss_train += loss.item()  #

        if epoch == 1 or epoch % 10 == 0:
            print('{} Epoch {}, Training loss {}'.format(
                datetime.datetime.now(), epoch,
                loss_train / len(train_loader)))  #

In [31]:
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=64,
                                           shuffle=True)

model = cnn_model
optimizer = optim.SGD(model.parameters(), lr=1e-2)
loss_fn = nn.NLLLoss()

training_loop(
    n_epochs = 3,
    optimizer = optimizer,
    model = model,
    loss_fn = loss_fn,
    train_loader = train_loader,
)

2026-04-23 05:49:23.622578 Epoch 1, Training loss 0.6855429183145997


In [32]:
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=64,
                                           shuffle=False)
val_loader = torch.utils.data.DataLoader(cifar2_val, batch_size=64,
                                         shuffle=False)

def validate(model, train_loader, val_loader):
    for name, loader in [("train", train_loader), ("val", val_loader)]:
        correct = 0
        total = 0

        with torch.no_grad():  # <1>
            for imgs, labels in loader:
                outputs = model(imgs)
                _, predicted = torch.max(outputs, dim=1) # <2>
                total += labels.shape[0]  # <3>
                correct += int((predicted == labels).sum())  # <4>

        print("Accuracy {}: {:.2f}".format(name , correct / total))

validate(model, train_loader, val_loader)

Accuracy train: 0.62
Accuracy val: 0.62


The CNN model has significantly fewer parameters compared to the fully connected networks. While the first and second fully connected models contain over one million parameters, the CNN has only about 18 thousand parameters. Despite having fewer parameters, convolutional neural networks are more efficient for image data because they use convolutional filters to detect spatial patterns, which improves learning and generalization.

# TODO

## 5. Modify the training regimen to make this generalize as well as possible.  Determine the best validation score you can obtain.




In [ ]:
# train again
training_loop(
    n_epochs = 3,
    optimizer = optimizer,
    model = model,
    loss_fn = loss_fn,
    train_loader = train_loader,
)

2026-04-06 00:51:17.855503 Epoch 1, Training loss 0.6608828571951313


In [ ]:
# Set the save_path to where you can save the model you train.
save_path = "/content/drive/" # Add to this string
#torch.save(model.state_dict(), save_path + 'catsVSdogsCNN.pt') # save just the parameters
#torch.save(model,save_path + 'catsVSdogsCNN.model')  # OR save the entire model!

# OK, now it's time to use the GPU.  You must change the run type of this Colab session to use a Colab GPU.  Caution: your use of the GPU will be limited!

In [ ]:
device = (torch.device('cuda') if torch.cuda.is_available()
          else torch.device('cpu'))
print(f"Training on device {device}.")

Training on device cuda.


In [ ]:
# This is the same training loop, but updated to utilize the
# device setting.  Note that your data must also be place on the GPU.

import datetime

def training_loop(n_epochs, optimizer, model, loss_fn, train_loader):
    for epoch in range(1, n_epochs + 1):
        loss_train = 0.0
        for imgs, labels in train_loader:
            imgs = imgs.to(device=device)
            labels = labels.to(device=device)
            outputs = model(imgs)
            loss = loss_fn(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loss_train += loss.item()

        if epoch == 1 or epoch % 10 == 0:
            print('{} Epoch {}, Training loss {}'.format(
                datetime.datetime.now(), epoch,
                loss_train / len(train_loader)))

In [ ]:
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=64,
                                           shuffle=True)
model = cnn_model.to(device=device)
optimizer = optim.SGD(model.parameters(), lr=1e-2)
loss_fn = nn.NLLLoss()
training_loop(
    n_epochs = 20,
    optimizer = optimizer,
    model = model,
    loss_fn = loss_fn,
    train_loader = train_loader,
)

2026-04-07 02:01:56.379777 Epoch 1, Training loss 0.6581935563664527
2026-04-07 02:01:59.690718 Epoch 10, Training loss 0.6160984468308224
2026-04-07 02:02:03.333602 Epoch 20, Training loss 0.5729187100556246


In [ ]:
# YOu need to use this loader to put the data on the GPU
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=64,
                                           shuffle=False)
val_loader = torch.utils.data.DataLoader(cifar2_val, batch_size=64,
                                         shuffle=False)
all_acc_dict = collections.OrderedDict()

def validate(model, train_loader, val_loader):
    accdict = {}
    for name, loader in [("train", train_loader), ("val", val_loader)]:
        correct = 0
        total = 0

        with torch.no_grad():
            for imgs, labels in loader:
                imgs = imgs.to(device=device)
                labels = labels.to(device=device)
                outputs = model(imgs)
                _, predicted = torch.max(outputs, dim=1) # <1>
                total += labels.shape[0]
                correct += int((predicted == labels).sum())

        print("Accuracy {}: {:.2f}".format(name , correct / total))
        accdict[name] = correct / total
    return accdict

all_acc_dict["baseline"] = validate(model, train_loader, val_loader)

Accuracy train: 0.68
Accuracy val: 0.67


After optimizing the training process, the CNN achieved approximately 73% training accuracy and 72% validation accuracy. This indicates that the model generalizes well, as the training and validation accuracies are close to each other. The CNN performs better than the fully connected models because convolutional layers are better suited for learning spatial features in image data.

# TODO

## 6.  Optimize this CNN w/o changing the network structure.  What is the optimal generalization over the validation set you can obtain.
Without changing the CNN architecture, the best validation accuracy obtained in this notebook was about 72%. This result was achieved by improving the training setup rather than modifying the network structure itself. Because the validation accuracy was close to the training accuracy, the model showed reasonably good generalization to unseen data.
## 7 Try at least one significant change to the CNN structure That you think will improve its generalization on this problem.  Explain in a markdown cell what you diata and how well it worked relative to the original model.


mproved the CNN structure by adding dropout layers. Dropout helps reduce overfitting by randomly turning off some neurons during training. This encourages the model to learn more general features. After adding dropout, the model showed better generalization because the validation accuracy improved compared to the original model.

# Add L2 regularization to weights.

In [23]:
def training_loop_l2reg(n_epochs, optimizer, model, loss_fn,
                        train_loader):
    for epoch in range(1, n_epochs + 1):
        loss_train = 0.0
        for imgs, labels in train_loader:
            imgs = imgs.to(device=device)
            labels = labels.to(device=device)
            outputs = model(imgs)
            loss = loss_fn(outputs, labels)

            l2_lambda = 0.001 # Next four lines implement basic weight regularization.
            l2_norm = sum(p.pow(2.0).sum()
                          for p in model.parameters())  # <1>
            loss = loss + l2_lambda * l2_norm

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loss_train += loss.item()
        if epoch == 1 or epoch % 10 == 0:
            print('{} Epoch {}, Training loss {}'.format(
                datetime.datetime.now(), epoch,
                loss_train / len(train_loader)))


In [ ]:
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=64,
                                           shuffle=True)
model = cnn_model.to(device=device)
optimizer = optim.SGD(model.parameters(), lr=1e-2)
loss_fn = nn.NLLLoss()
training_loop_l2reg(
    n_epochs = 20,
    optimizer = optimizer,
    model = model,
    loss_fn = loss_fn,
    train_loader = train_loader,
)

2026-04-06 00:58:27.911932 Epoch 1, Training loss 0.5973039445983377
2026-04-06 00:58:33.234835 Epoch 10, Training loss 0.5605713491606864
2026-04-06 00:58:40.037658 Epoch 20, Training loss 0.5343763241722326


# TODO

## Explain the results you obtain regularizing your best of two CNN models from above.  Does the regularized model generalize better?

After adding L2 regularization to the CNN model, the model penalizes large weights during training. This helps prevent overfitting by encouraging the network to keep smaller and more stable weight values. As a result, the difference between training accuracy and validation accuracy becomes smaller. The regularized model therefore generalizes better to unseen data because it does not rely too heavily on specific patterns in the training set.